# FLEO — Local training (VS Code + GPU)

**Trains YOLOv8n + FLEO on FER2013 and RAF-DB** (DPU-native backbone, so it compiles to the ZCU104 later).

1. Top-right **Select Kernel** -> pick your Python (Global 3.10/3.11).
2. **Run All**.

Tuned for a 6 GB GPU (RTX 2060): `yolov8n`, `imgsz 128`, `batch 16`, `40 epochs`.
Total ~3.5-4.5 h for all four runs. If you hit `CUDA out of memory`, drop `--batch` to 8.

## 1. Install dependencies (CUDA build of PyTorch + ultralytics)

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
%pip install -q ultralytics onnx onnxruntime pyyaml

## 2. Move to the repo root and check the GPU

In [ ]:
import os
# The notebook lives in notebooks/; step up to the repo root (which has scripts/ and fleo/).
if not os.path.isdir('scripts'):
    os.chdir('..')
print('cwd:', os.getcwd())
assert os.path.isdir('scripts') and os.path.isdir('fleo'), 'Open this notebook from inside the FLEO repo.'

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'No GPU visible to torch -- reinstall the CUDA build (cell 1) or check drivers.'

## 3. Fix the dataset paths (they may still point at Kaggle)

In [ ]:
import re, pathlib
for ds in ['fer2013', 'rafdb']:
    p = pathlib.Path(f'datasets/{ds}/data.yaml')
    if p.exists():
        local = pathlib.Path(f'datasets/{ds}').resolve().as_posix()
        t = re.sub(r'^path:.*', f'path: {local}', p.read_text(), flags=re.M)
        p.write_text(t)
        print(ds, '->', t.splitlines()[0])
    else:
        print('MISSING (skip):', p)

## 4. Train the four models
Order: RAF-DB first (smaller/faster, cleaner data), then FER2013. Each run saves to `runs/fleo/<variant>_seed0/weights/best.pt`.

In [ ]:
import subprocess, sys

RUNS = [
    ('rafdb',   'fleo'),
    ('rafdb',   'baseline'),
    ('fer2013', 'fleo'),
    ('fer2013', 'baseline'),
]

for ds, variant in RUNS:
    yml = f'datasets/{ds}/data.yaml'
    if not pathlib.Path(yml).exists():
        print(f'skip {ds}/{variant}: {yml} not found'); continue
    print(f'\n===================  {ds} / {variant}  ===================', flush=True)
    r = subprocess.run([
        sys.executable, '-m', 'scripts.train',
        '--data', yml, '--variant', variant,
        '--cfg', 'yolov8n.yaml', '--pretrained', 'yolov8n.pt',
        '--epochs', '40', '--imgsz', '128', '--batch', '16',
        '--device', '0', '--seeds', '0', '--workers', '2',
    ])
    print(f'{ds}/{variant} finished with exit code {r.returncode}', flush=True)

## 5. Next steps (after training)
The four `best.pt` weights are under `runs/fleo/`. Then, inside the **Vitis AI docker**:
```bash
python -m scripts.export --weights runs/fleo/fleo_seed0/weights/best.pt --route r1 --imgsz 128 --verify
PYTHONPATH=/work python deploy/quantize_vai.py --weights export/r1.pt --route r1 \
    --cfg yolov8n.yaml --data datasets/fer2013/data.yaml --imgsz 128 --mode test --out quantized/r1
bash deploy/compile_b4096.sh r1 quantized/r1 compiled/r1
xdputil xmodel compiled/r1/*.xmodel -l   # expect a single DPU subgraph (no area-attention)
```